In [8]:
import sys
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaLLM
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [9]:
# ==========================================
# 1. LLM Setup
# ==========================================
print("🔄 Initializing Ollama (Llama 3.1)...")
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
    timeout=60  
)

🔄 Initializing Ollama (Llama 3.1)...


In [10]:
#connect cromadb vector db
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  # embedding তৈরির সময় যেটা ব্যবহার করেছিলে
)

vector_db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9790.67it/s]


In [ ]:
chat_history = []


def add_to_history(user, assistant):
    chat_history.append({"user": user, "assistant": assistant})

In [ ]:
template = """You are a helpful assistant answering questions based strictly on the provided context. 
    If you don't know the answer or if it's not in the context, say that you don't know. 
    Do not make things up.

    chat_history: {chat_history}

    Context:
    {context}

    Question: 
    {question}

    Answer:"""
prompt = ChatPromptTemplate.from_template(template)

In [12]:
#Helper function to format retrieved documents nicely for the prompt
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [ ]:


def run_rag_pipeline(user_question):
  retriever = vector_db.as_retriever(search_kwargs={"k": 3})
  #Construct the LangChain RAG Chain
  rag_chain = (
      {"context": retriever | format_docs, "question": RunnablePassthrough()}
      | prompt
      | llm
      | StrOutputParser()
  )
  #Execute the chain
  print(f"\nQuerying: '{user_question}'\n")
  response = rag_chain.invoke(user_question)

  add_to_history(user_question, response)
    
  print("--- AI Answer ---")
  print(response)

In [3]:
query = "who is abu sufiun?"
run_rag_pipeline(query)

NameError: name 'vector_db' is not defined

In [15]:
run_rag_pipeline("how many skill he has?")


Querying: 'how many skill he has?'

--- AI Answer ---
According to the provided context, Abu Sufiun has 13 skills listed under "SKILLS":

1. RESTful API
2. Laravel
3. React
4. Bootstrap
5. Tailwind CSS
6. MS-SQL
7. Oracle
8. My-SQL
9. OOP (Object-Oriented Programming)
10. PHP
11. JavaScript
12. C
13. AI (Artificial Intelligence) and ML (Machine Learning)

So, the answer is 13.


In [4]:
# NEW & WORKING
from langchain_classic import hub